# Run an active-learning sampler

One sampler, one dataset, the full budget sweep. Several configs of the same
sampler can run back to back in one session so the dataset and the DINOv2 cache
load once.

Per budget this writes selected indices, the probe weights, the metrics table,
the PALM fit and a run log — see `main.py`. The last cell zips all of it.

In [ ]:
from pathlib import Path
import subprocess
import sys

REPO_URL = "https://github.com/CryAndRRich/codapath.git"
REPO_BRANCH = "namhai"
REPO = Path("/kaggle/working/codapath")

if (REPO / ".git").is_dir():
    subprocess.check_call(["git", "-C", str(REPO), "fetch", "origin", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "switch", REPO_BRANCH])
    subprocess.check_call(["git", "-C", str(REPO), "pull", "--ff-only", "origin", REPO_BRANCH])
elif REPO.exists():
    raise RuntimeError(f"{REPO} exists but is not a Git repository")
else:
    subprocess.check_call(
        ["git", "clone", "--branch", REPO_BRANCH, "--single-branch", REPO_URL, str(REPO)]
    )

branch = subprocess.check_output(
    ["git", "-C", str(REPO), "branch", "--show-current"], text=True
).strip()
assert branch == REPO_BRANCH, (branch, REPO_BRANCH)
print("repo:", REPO, "| branch:", branch)

In [ ]:
%cd /kaggle/working/codapath

In [ ]:
import os
import sys

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-U", "huggingface_hub", "hf-transfer"])

os.environ["TOKENIZERS_PARALLELISM"] = "false"
if "/kaggle/working/codapath" not in sys.path:
    sys.path.append("/kaggle/working/codapath")

In [ ]:
# ---- EDIT THIS CELL ----
DATASET = "pathmnist"

# Any name in sampling/specs.py:
#   random | coreset | codapath | typiclust | activeft | tcm
#   margin | entropy | badge | dropquery | uncertainty_herding | refine
#   scalpel   <- this project's method
SAMPLER = "scalpel"
SEED = 42

# One dict per full budget sweep. Use [{}] to run the config file as-is.
#
# scalpel axes (see config/config.yaml for the full list):
#   {}                                     the main method: visual/cell disagreement
#   {"uncertainty_mode": "visual_margin"}  ablation: plain Uncertainty Herding weight
#   {"cell_pooling": "rff"}                kernel mean embedding instead of a cell mean
#   {"missing_impute": "zero"}             ablation for patches with no nucleus
#   {"consistency_weight": 0.1}            couples the two probes while training
VARIANTS = [
    {},
    {"uncertainty_mode": "visual_margin"},
]

# Leave RUN_NAME None so each variant gets a collision-safe name of its own.
RUN_NAME = None

# Starting points only; the next cell searches for the real directories.
FEATURE_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/features"
CELLVIT_DIR = "/kaggle/input/datasets/cryandrrich/nckh2026/cellvit_features"
OUTPUT_DIR = "/kaggle/working/checkpoints"

In [ ]:
from huggingface_hub import snapshot_download

print("Downloading facebook/dinov2-base ...")
snapshot_download(repo_id="facebook/dinov2-base")

In [ ]:
import yaml
import torch

import main
from sampling.specs import spec_for

In [ ]:
DATA_ROOT_CANDIDATES = [
    Path("/kaggle/input/datasets/cryandrrich/nckh2026"),
    Path("/kaggle/input/nckh2026"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), DATA_ROOT_CANDIDATES[0])

DATA_PATHS = {
    "pathmnist": str(DATA_ROOT / "pathmnist_224.npz"),
    "histoset": str(DATA_ROOT / "HistoSet-5x14/HistoSet-5x14"),
    "skintissue": str(DATA_ROOT / "SkinTissue/SkinTissue/tiles"),
}
print("data root:", DATA_ROOT)

In [ ]:
with open("config/config.yaml", "r", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)

dataset_info = config["datasets"][DATASET]
training_cfg = config["training"]
base_cfg = dict(config.get("samplers", {}).get(SAMPLER, {}))
sampler_cfgs = [{**base_cfg, **overrides} for overrides in VARIANTS]
spec = spec_for(SAMPLER)

data_path = Path(DATA_PATHS[DATASET])
assert data_path.exists(), f"Missing Kaggle input: {data_path}"
assert torch.cuda.is_available(), "Attach a Kaggle GPU before running AL"
assert VARIANTS, "VARIANTS needs at least one dict (use [{}])"
assert RUN_NAME is None or len(VARIANTS) == 1, (
    "A fixed RUN_NAME with several variants makes every run overwrite the previous one"
)

SEARCH_ROOTS = [DATA_ROOT, Path("/kaggle/input")]


def dir_containing(probe, hint=None, max_depth=3):
    """Return the directory D such that D/probe exists.

    Publishing /kaggle/working/<name> as a Kaggle Dataset remounts it as
    /kaggle/input/<slug>/<name>/<name>, one level deeper than the path anyone
    writes down, so a hard-coded cache path fails in a way that is tedious to
    debug from a stack trace. Search a few levels and print what was found.
    """
    probe = Path(probe)
    up = len(probe.parts) - 1
    roots = ([Path(hint)] if hint else []) + SEARCH_ROOTS
    for root in roots:
        if not root.exists():
            continue
        for depth in range(max_depth + 1):
            pattern = "/".join(["*"] * depth + list(probe.parts))
            for hit in sorted(root.glob(pattern)):
                return hit.parents[up]
    return None


# Only samplers that declare a cell view need the CellViT cache.
if "cell_embeddings" in spec.needs:
    found = dir_containing(f"{DATASET}_seed{SEED}/manifest.json", CELLVIT_DIR)
    assert found is not None, (
        f"No CellViT cache for {DATASET}_seed{SEED} under {CELLVIT_DIR} or "
        f"{[str(r) for r in SEARCH_ROOTS]}. Attach the extraction dataset."
    )
    CELLVIT_DIR = str(found)
    print("cellvit cache:", CELLVIT_DIR)

# The DINOv2 cache is optional: main.py re-extracts on a miss. It must not
# re-extract into a read-only /kaggle/input, which would only fail AFTER the
# whole forward pass, so fall back to a writable directory instead.
vit_name = config.get("models", {}).get("vit", "facebook/dinov2-base")
safe_vit = vit_name.replace("/", "_")
found = dir_containing(f"{DATASET}_seed{SEED}_{safe_vit}_train.npy", FEATURE_DIR)
manifest_name = f"{DATASET}_seed{SEED}_{safe_vit}_manifest.json"
if found is None:
    print(f"[features] not found for {DATASET}/seed{SEED}/{vit_name} - will extract this session.")
    FEATURE_DIR = "/kaggle/working/features"
elif not (found / manifest_name).is_file():
    print(f"[features] found .npy at {found} but no {manifest_name}.")
    print("  A cache without a sample-order manifest is rejected (row alignment")
    print("  cannot be verified), so it will be extracted once this session.")
    FEATURE_DIR = "/kaggle/working/features"
else:
    FEATURE_DIR = str(found)
    print("features cache:", FEATURE_DIR)

if not str(FEATURE_DIR).startswith("/kaggle/input"):
    Path(FEATURE_DIR).mkdir(parents=True, exist_ok=True)
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

print(f"{SAMPLER}: {spec.passes} pass, prefix_exact={spec.prefix_exact} - {spec.why}")
for cfg in sampler_cfgs:
    print("  config:", cfg)

In [ ]:
import time

for index, sampler_cfg in enumerate(sampler_cfgs, start=1):
    started = time.time()
    print("=" * 70)
    print(f"VARIANT {index}/{len(sampler_cfgs)}: {VARIANTS[index - 1]}")
    print("=" * 70)
    main.run(
        data_path=str(data_path),
        sampler_name=SAMPLER,
        num_classes=dataset_info["num_classes"],
        cumulative_budget=config["cumulative_budget"],
        data_descriptions=dataset_info.get("descriptions", {}),
        prompt_templates=config.get("prompt_templates", []),
        sampler_cfg=sampler_cfg,
        probe_epochs=training_cfg["probe_epochs"],
        probe_lr=training_cfg["probe_lr"],
        device=torch.device(config["device"]),
        random_seed=SEED,
        save_dir=str(Path(OUTPUT_DIR) / DATASET),
        verbose=True,
        model_cfg=config.get("models", {}),
        feature_cache_dir=FEATURE_DIR,
        cellvit_cache_dir=CELLVIT_DIR,
        run_name=RUN_NAME,
    )
    print(f"VARIANT {index} finished in {(time.time() - started) / 60:.1f} min")

In [ ]:
# Zip everything this notebook produced so it downloads as one file.
import shutil

SOURCE = Path('/kaggle/working/checkpoints')
ARCHIVE = Path("/kaggle/working/al_checkpoints")
assert SOURCE.is_dir(), f"nothing to archive at {SOURCE}"
shutil.make_archive(str(ARCHIVE), "zip", root_dir=SOURCE)
size_mb = ARCHIVE.with_suffix(".zip").stat().st_size / 1e6
print(f"{ARCHIVE}.zip  ({size_mb:.1f} MB)")